# 02 - No-show prediction (CRISP-DM phases 4 and 5)

This notebook takes over `Prediction_No_Show.ipynb` of the conference paper: same split, same SMOTE, same four models and hyperparameter grids, same metrics computed the same way (`utils_ml.py`). The changes are announced in a **Correction** box where they act:

1. `discontinued` and `service_time` are not predictors any more (notebook 01, section 3.6).
2. SMOTE and the scaler are inside the cross-validation (a pipeline), so that the synthetic no-shows of a fold are never used to score that fold.
3. The decision threshold is chosen on a validation set (part of the training set), never on the test set.
4. The MLP stops on its own validation data, not on the test set.
5. The probabilities are brought back to the real no-show rate before the optimisation (notebook 04).
6. The categorical variables with many values (zip, sta3n, state, stopcode) are target-encoded instead of integer codes, whose order means nothing (zip 437 is not "between" 436 and 438).

## Outline
- [0 - Parameters](#0)
- [1 - Data and split](#1)
- [2 - Models](#2): [Random Forest](#2.1), [XGBoost](#2.2), [AdaBoost](#2.3), [MLP](#2.4)
- [3 - Comparison of the models](#3)
- [4 - Probabilities for the scheduling model](#4)

In [ ]:
import sys
sys.path.insert(0, "..")          # utils_ml.py is at the root of the repository

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from utils_ml import *

<a name="0"></a>
## 0 - Parameters

Every choice of this notebook is set in the cell below, with its reason.

In [ ]:
DATA_PATH = "../data/data_prep.parquet"      # written by 01_data_preparation.ipynb
SPLIT_PATH = "../data/split.parquet"         # train / validation / test of each consultation, reused by notebook 03
SCORES_PATH = "../data/test_scores.parquet"  # no-show probabilities of the test consultations, read by notebook 04
TARGET = "NoShow"
NOT_PREDICTORS = ["service_time"]             # target of notebook 03, known only after the consultation

# Share of the consultations used: 0.1 for a pilot run, 1.0 for the final run.
SAMPLE_FRACTION = 1.0
RANDOM_STATE = 0                              # as in the paper

# Categorical variables with many values (zip 910, sta3n 130, state 66, stopcode 36). Each value is
# replaced by the no-show rate observed for it in the train set (target encoding, smoothed towards
# the overall rate for rare values). On the train set it is computed by 5-fold cross-fitting, so a
# consultation never sees its own outcome; validation and test use the rates of the whole train set.
CATEGORICAL = ["sta3n", "stopcode", "zip", "state"]
TARGET_ENCODER = dict(target_type="binary", smooth="auto", cv=5, shuffle=True, random_state=RANDOM_STATE)

TEST_SIZE = 0.3                               # as in the paper: 70 % train, 30 % test, stratified
VALIDATION_SIZE = 0.15                        # Correction: share of the train set kept aside to choose the threshold
SMOTE_RATIO = 0.33                            # as in the paper: after SMOTE, no-shows = 33 % of the shows (train only)
CV_FOLDS = 3                                  # as in the paper
SCORING = "f1_macro"                          # the models are optimised for the F1 score, averaged as in the paper's metrics
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)   # candidate decision thresholds

# Hyperparameter grids of the paper (Prediction_No_Show.ipynb).
GRID_RF = {"n_estimators": [100], "max_depth": [10, 15, 20], "min_samples_split": [2, 5],
           "min_samples_leaf": [2, 4], "max_features": ["sqrt"], "class_weight": ["balanced"]}
GRID_XGB = {"n_estimators": [1000], "max_depth": [20, 25], "min_child_weight": [2], "gamma": [0.3],
            "subsample": [0.4], "reg_alpha": [0.2], "learning_rate": [0.01, 0.1]}
XGB_SCALE_POS_WEIGHT = 1 / SMOTE_RATIO        # as in the paper: shows / no-shows after SMOTE
GRID_ADA = {"n_estimators": [100], "learning_rate": [0.05, 0.1]}
# MLP of the paper: 16-8-4 ReLU, Adam, early stopping with patience 10, at most 50 epochs.
# scikit-learn instead of Keras (one dependency less); no dropout layer in scikit-learn, an L2
# penalty (alpha) plays that role; batch of 256 instead of 32 (10 M rows: 8 times fewer steps).
MLP_PARAMS = dict(hidden_layer_sizes=(16, 8, 4), activation="relu", solver="adam", alpha=1e-4,
                  batch_size=256, max_iter=50, early_stopping=True, validation_fraction=0.1,
                  n_iter_no_change=10, random_state=RANDOM_STATE)

# No-show rate each model is trained at (for the correction of its probabilities, section 4):
# class_weight='balanced' and scale_pos_weight give both classes the same weight (rate 0.5);
# otherwise the rate after SMOTE, 0.33 / 1.33.
TRAINING_RATE = {"Random Forest": 0.5, "XGBoost": 0.5,
                 "AdaBoost": SMOTE_RATIO / (1 + SMOTE_RATIO), "MLP": SMOTE_RATIO / (1 + SMOTE_RATIO)}

PERMUTATION_ROWS = 200_000                    # rows for the permutation importance of the MLP
RELIABILITY_BINS = 20                         # bins of the reliability diagrams

# Test metrics published in the conference paper (Prediction_No_Show.ipynb), for comparison.
PAPER_TEST_METRICS = pd.DataFrame(
    {"Random Forest": [0.707852, 0.830735, 0.898758, 0.568598, 0.591063],
     "XGBoost":       [0.669060, 0.864867, 0.967843, 0.727743, 0.693618],
     "AdaBoost":      [0.527527, 0.814743, 0.967831, 0.676746, 0.542222],
     "MLP":           [0.640944, 0.797446, 0.950434, 0.616585, 0.627397]},
    index=['Recall', 'AUC', 'Accuracy', 'Precision', 'F1 Score'])

<a name="1"></a>
## 1 - Data and split

In [ ]:
data = pd.read_parquet(DATA_PATH)
if SAMPLE_FRACTION < 1:
    data = data.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE)
# The categorical variables first: the encoder of the pipeline outputs them first.
FEATURES = CATEGORICAL + [c for c in data.columns if c not in CATEGORICAL + [TARGET] + NOT_PREDICTORS]
print(f"{len(data):,} consultations, no-show rate {data[TARGET].mean():.4f}")
print(f"{len(FEATURES)} predictors: {FEATURES}")
data[CATEGORICAL].nunique().to_frame("distinct values")

As in the paper: 70 % train, 30 % test, stratified on the target.

> **Correction.** The paper chose the decision threshold by looking at the test set (0.6 for the Random Forest, 0.49 for AdaBoost). Here 15 % of the train set is kept aside as a **validation** set: the threshold is chosen there, and the test set is only used once, to report the metrics.

In [ ]:
rows_train, rows_test = train_test_split(data.index, test_size=TEST_SIZE, random_state=RANDOM_STATE,
                                         stratify=data[TARGET], shuffle=True)
rows_train, rows_val = train_test_split(rows_train, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE,
                                        stratify=data.loc[rows_train, TARGET])
split = pd.concat([pd.Series("train", index=rows_train), pd.Series("validation", index=rows_val),
                   pd.Series("test", index=rows_test)]).rename("part").rename_axis("row").sort_index()
split.to_frame().to_parquet(SPLIT_PATH)

X_train, y_train = data.loc[rows_train, FEATURES], data.loc[rows_train, TARGET].to_numpy()
X_val, y_val = data.loc[rows_val, FEATURES], data.loc[rows_val, TARGET].to_numpy()
X_test, y_test = data.loc[rows_test, FEATURES], data.loc[rows_test, TARGET].to_numpy()
summary = pd.DataFrame({"rows": [len(y_train), len(y_val), len(y_test)],
                        "no-shows": [y_train.sum(), y_val.sum(), y_test.sum()]},
                       index=["train", "validation", "test"])
summary["no-show rate"] = summary["no-shows"] / summary["rows"]
summary

<a name="2"></a>
## 2 - Models

> **Correction.** In the paper SMOTE was applied to the whole train set before the cross-validation: the synthetic no-shows built from a consultation could then be scored in another fold, which flatters the cross-validation scores. Here the target encoder, then SMOTE, then the MinMaxScaler, then the model form one pipeline (`make_pipeline`): in each fold they only see the training part of the fold. The best pipeline is refit the same way on the whole train set; validation and test are never resampled.

For each model: grid search (`train_model`), then the threshold chosen on the validation set, the metrics of the paper on train and test, the ROC curve, the errors and the feature importances (`evaluate_classifier`).

In [ ]:
models, probas, thresholds, results = {}, {}, {}, {}

def run(name, model, grid):
    pipeline = make_pipeline(model, CATEGORICAL, TARGET_ENCODER, SMOTE_RATIO, RANDOM_STATE)
    models[name], cv, minutes = train_model(pipeline, grid, X_train, y_train, SCORING, CV_FOLDS, RANDOM_STATE)
    print(f"{name}: trained in {minutes:.1f} min")
    if cv is not None:
        print("Best hyperparameters:", cv.iloc[0]["params"])
        display(cv)
    probas[name] = {part: models[name].predict_proba(X)[:, 1]
                    for part, X in [("train", X_train), ("val", X_val), ("test", X_test)]}

def evaluate(name):
    thresholds[name], results[name] = evaluate_classifier(
        name, models[name], probas[name], y_train, y_val, y_test, X_test, THRESHOLDS,
        FEATURES, PERMUTATION_ROWS, RANDOM_STATE)
    display(results[name].round(4))

<a name="2.1"></a>
### 2.1 - Random Forest

In [ ]:
run("Random Forest", RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE), GRID_RF)
evaluate("Random Forest")

<a name="2.2"></a>
### 2.2 - XGBoost

In [ ]:
run("XGBoost", xgb.XGBClassifier(objective="binary:logistic", scale_pos_weight=XGB_SCALE_POS_WEIGHT,
                                 tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE), GRID_XGB)
evaluate("XGBoost")

<a name="2.3"></a>
### 2.3 - AdaBoost

In [ ]:
run("AdaBoost", AdaBoostClassifier(random_state=RANDOM_STATE), GRID_ADA)
evaluate("AdaBoost")

<a name="2.4"></a>
### 2.4 - MLP

> **Correction.** In the paper the MLP stopped when its loss on the **test** set stopped improving (`validation_data=(X_test, y_test)`), so the test set took part in the training. Here it stops on 10 % of its own training data (`early_stopping=True`).

In [ ]:
run("MLP", MLPClassifier(**MLP_PARAMS), None)
mlp = models["MLP"].named_steps["model"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(mlp.loss_curve_); ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(mlp.validation_scores_); ax[1].set_title("Accuracy on its validation data"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()
print(f"Stopped after {mlp.n_iter_} epochs")
evaluate("MLP")

<a name="3"></a>
## 3 - Comparison of the models

Test metrics of the four models, next to the metrics of the conference paper. The differences come from the changes listed at the top (mainly the two predictors removed, which were known only after the consultation).

In [ ]:
test_metrics = pd.DataFrame({name: res["Test"] for name, res in results.items()})
comparison = pd.concat({"this study": test_metrics, "conference paper": PAPER_TEST_METRICS[test_metrics.columns]}, axis=1)
comparison = comparison.swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)
display(comparison.round(4))
test_metrics.loc["Threshold"] = pd.Series(thresholds)
test_metrics.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
test_metrics.drop("Threshold").plot.bar(ax=ax, rot=0)
ax.set_ylim(0, 1); ax.set_title("Test metrics"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The model used for the scheduling is the one with the best F1 score on the **validation** set (the test set does not take part in the choice).

In [ ]:
val_f1 = pd.Series({name: f1_score(y_val, (probas[name]["val"] >= thresholds[name]).astype(int), average="macro")
                    for name in models})
print("F1 (macro) on the validation set:"); print(val_f1.round(4).to_string())
BEST = val_f1.idxmax()
print(f"\nModel used for the scheduling: {BEST}")

<a name="4"></a>
## 4 - Probabilities for the scheduling model

The scheduling model (notebook 04) needs the **probability** that each patient comes, not only a yes/no prediction. The models were trained on data where no-shows are much more frequent than in reality (SMOTE, class weights), so their probabilities are too high. Bayes' rule brings them back to the real no-show rate (`prior_correction`, one formula, Elkan 2001). The reliability diagram below, on the validation set, shows the probabilities before and after: well-calibrated probabilities lie on the diagonal.

In [ ]:
true_rate = y_train.mean()
print(f"Real no-show rate (train): {true_rate:.4f}; rate {BEST} was trained at: {TRAINING_RATE[BEST]:.4f}")
val_corrected = prior_correction(probas[BEST]["val"], TRAINING_RATE[BEST], true_rate)
plot_reliability(y_val, {"model output": probas[BEST]["val"], "after prior correction": val_corrected},
                 RELIABILITY_BINS, f"{BEST}: reliability on the validation set")
print(f"Mean probability on validation: model {probas[BEST]['val'].mean():.4f}, "
      f"corrected {val_corrected.mean():.4f}, observed rate {y_val.mean():.4f}")

Export, for every consultation of the test set (identified by its row in `data_prep.parquet`): its corrected no-show probability and whether it was actually a no-show.

In [ ]:
test_scores = pd.DataFrame({
    "row": rows_test,
    "p_noshow": prior_correction(probas[BEST]["test"], TRAINING_RATE[BEST], true_rate),
    "NoShow": y_test,
})
test_scores.to_parquet(SCORES_PATH, index=False)
print(f"Saved {SCORES_PATH}: {len(test_scores):,} test consultations, model {BEST}")
test_scores.drop(columns="row").describe()